# **PDG  - TIME UP AND GO**
### Predecesores.
*   **Juan David Patiño**
*   **Daniel Montezuma**
*  **Diego Mueses**

### Actuales.
* **Esteban Guarin Valencia**
* **Santiago Gomez Robledo**
* **Heiner Danit Rincon**
* **Juan Jose Arias Gallego**

# Codigo PDG Predecesor

## **Código**

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def cargar_datos_json(ruta_archivo):
    """Carga datos desde un archivo JSON."""
    try:
        with open(ruta_archivo, 'r') as file:
            return json.load(file)
    except FileNotFoundError:
        print(f"Error: No se encontró el archivo '{ruta_archivo}'")
        return None
    except json.JSONDecodeError:
        print(f"Error: El archivo '{ruta_archivo}' no contiene un JSON válido")
        return None

## **Graficar Señales Base**

In [ ]:

import json
import matplotlib.pyplot as plt
import numpy as np

def plot_json_data_with_phases(json_data, timestamp_index=None, timestamp_value=None,
                               phases=None, colors=None, alpha=0.2):
    """
    Grafica los datos de un JSON, permitiendo filtrar por índice o valor específico de timestamp.
    Resalta fases del movimiento con colores de fondo diferentes.

    Parámetros:
    - json_data: Datos JSON a procesar
    - timestamp_index: Índice (posición) del timestamp a procesar (opcional)
    - timestamp_value: Valor específico del timestamp a procesar (opcional)
    - phases: Lista de diccionarios con las fases a resaltar. Cada fase debe tener:
        - 'name': Nombre de la fase
        - 'start': Tiempo de inicio de la fase
        - 'end': Tiempo de fin de la fase
        - 'color': (opcional) Color específico para esta fase
    - colors: Lista de colores para las fases (si no se especifica un color por fase)
    - alpha: Transparencia para las áreas coloreadas
    """
    # Valores por defecto
    if phases is None:
        phases = []

    if colors is None:
        colors = ['lightpink', 'lightgreen', 'lightblue', 'lightyellow', 'lightcyan']

    # Seleccionar timestamps a procesar
    timestamps = list(json_data.keys())
    timestamps_to_process = []

    if timestamp_index is not None:
        if 0 <= timestamp_index < len(timestamps):
            timestamps_to_process = [timestamps[timestamp_index]]
        else:
            print(f"Error: Índice {timestamp_index} fuera de rango. Rango válido: 0-{len(timestamps)-1}")
            return
    elif timestamp_value is not None:
        if timestamp_value in json_data:
            timestamps_to_process = [timestamp_value]
        else:
            print(f"Error: Timestamp '{timestamp_value}' no encontrado en los datos")
            return
    else:
        timestamps_to_process = timestamps

    for timestamp in timestamps_to_process:
        print(f"Procesando timestamp: {timestamp}")
        prueba = json_data[timestamp]

        for lado, datos in prueba.items():
            tiempos = [d["t"] for d in datos]
            valores = {key: [d[key] for d in datos] for key in ["a", "b", "g", "x", "y", "z"]}

            plt.figure(figsize=(15, 6))

            # Resaltar las fases del movimiento
            for i, phase in enumerate(phases):
                color = phase.get('color', colors[i % len(colors)])
                plt.axvspan(phase['start'], phase['end'], alpha=alpha, color=color, label=phase['name'])

                # Líneas verticales al inicio y fin de cada fase
                plt.axvline(x=phase['start'], color='gray', linestyle='--', alpha=0.7)
                plt.axvline(x=phase['end'], color='gray', linestyle='--', alpha=0.7)

                # Añadir etiquetas
                y_min, y_max = plt.ylim()
                plt.text(phase['start'] + (phase['end'] - phase['start'])/2, y_max * 0.95,
                         phase['name'], horizontalalignment='center', verticalalignment='top',
                         fontsize=9, bbox=dict(facecolor='white', alpha=0.6))

            # Graficar todos los valores
            for key, vals in valores.items():
                plt.plot(tiempos, vals, label=key, linewidth=1)

            plt.xlabel("Tiempo (ms)")
            plt.ylabel("Valores")
            plt.title(f"Datos de '{lado}' en {timestamp} - Fases de Movimiento")
            plt.legend(loc='upper right')
            plt.grid(True)
            plt.tight_layout()
            plt.show()


## **Transformada de Fourier (FFT)**

In [ ]:
# TRANSFORMADA DE FOURIER (FFT)
import json
import numpy as np
import matplotlib.pyplot as plt

def fft_json_data(json_data, timestamp_index=None, fs=50):  # fs: frecuencia de muestreo en Hz
    # Convertir las claves de timestamp a una lista ordenada
    timestamps = sorted(json_data.keys())

    # Si se especifica un índice de timestamp, usar solo ese
    if timestamp_index is not None:
        if 0 <= timestamp_index < len(timestamps):
            selected_timestamp = timestamps[timestamp_index]
            timestamps = [selected_timestamp]
            print(f"Analizando el timestamp {selected_timestamp} (índice {timestamp_index})")
        else:
            print(f"Índice de timestamp no válido. Debe estar entre 0 y {len(timestamps)-1}")
            return

    # Proceder con el análisis de los timestamps seleccionados
    for timestamp in timestamps:
        prueba = json_data[timestamp]
        for lado, datos in prueba.items():
            valores = {key: [d[key] for d in datos] for key in ["a", "b", "g", "x", "y", "z"]}

            plt.figure(figsize=(12, 6))
            for key, señal in valores.items():
                señal_np = np.array(señal)
                N = len(señal_np)

                # Aplicar FFT
                fft_result = np.fft.fft(señal_np)
                fft_freqs = np.fft.fftfreq(N, d=1/fs)

                # Tomar solo frecuencias positivas
                idx = np.where(fft_freqs >= 0)
                fft_magnitude = np.abs(fft_result[idx])
                fft_freqs = fft_freqs[idx]

                plt.plot(fft_freqs, fft_magnitude, label=key)

            plt.title(f"FFT - '{lado}' en {timestamp}")
            plt.xlabel("Frecuencia (Hz)")
            plt.ylabel("Magnitud")
            plt.legend()
            plt.grid()
            plt.tight_layout()
            plt.show()


In [ ]:
def graficar_energia_y_derivada_con_fases(
        json_data, señal_key="a", sensor="derecha",
        tamaño_ventana=1.0, paso=0.1, fs=25,
        timestamp_index=None, aplicar_window=True,
        phases=None, alpha=0.2):

    # Obtener datos originales
    pruebas_data = json_data["pruebas"]
    timestamps = sorted(pruebas_data.keys())

    if timestamp_index is not None:
        timestamps = [timestamps[timestamp_index]]

    timestamp = timestamps[0]
    datos_sensor = pruebas_data[timestamp][sensor]

    señal_original = [d[señal_key] for d in datos_sensor if señal_key in d]
    tiempos_original = [d["t"]/1000.0 for d in datos_sensor]

    # FFT deslizante
    tiempos_fft, energia_fft = analisis_fourier_ventana_deslizante(
        json_data, señal_key, sensor, tamaño_ventana,
        paso, fs, timestamp_index, aplicar_window
    )

    # --- SOLO 2 SUBPLOTS ---
    fig, axs = plt.subplots(2, 1, figsize=(15, 10), sharex=True)

    # --- Señal original ---
    axs[0].plot(tiempos_original, señal_original, 'b-', linewidth=1)
    axs[0].set_ylabel("Amplitud")
    axs[0].set_title(f"Señal Original - {sensor}/{señal_key}")
    axs[0].grid(True, linestyle='--', alpha=0.4)

    if phases:
        agregar_fases_al_grafico(axs[0], phases, alpha)

    # --- Energía FFT ---
    axs[1].plot(tiempos_fft, energia_fft, 'r-', linewidth=2, label="Energía espectral (FFT)")
    axs[1].set_ylabel("Energía")
    axs[1].set_xlabel("Tiempo (s)")
    axs[1].set_title(f"Energía Espectral (FFT) - Ventana: {tamaño_ventana}s, Paso: {paso}s")
    axs[1].grid(True, linestyle='--', alpha=0.4)
    axs[1].legend()

    if phases:
        agregar_fases_al_grafico(axs[1], phases, alpha)

    plt.tight_layout()
    plt.show()


## **Muestra**

In [ ]:
# señales disponibles ['a', 'b', 'g', 'x','y','z']
# sensores disponibles ['derecha', 'izquierda', 'espina_base']

ruta_archivo_TUG = "/20250709003.json"
datos_TUG = cargar_datos_json(ruta_archivo_TUG)

print("DATOS DE LA PRUEBA TUG")
print(datos_TUG)

ruta_archivo_FASES = "/20250709003.2.json"
datos_FASES = cargar_datos_json(ruta_archivo_FASES)

print("DATOS DE LAS FASES")
print(datos_FASES)

graficar_energia_y_derivada_con_fases(
                  datos_TUG,
                  señal_key='g',
                  sensor='espina_base',
                  tamaño_ventana=1.0,
                  paso=0.5,
                  fs=25,
                  timestamp_index=1,
                  aplicar_window=True,
                  phases=datos_FASES,  # Pasar las fases aquí
                  alpha=0.2
              )

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## **Domiciano**

In [ ]:
from scipy.signal import butter, filtfilt
import numpy as np
from scipy.signal import windows
from scipy.fft import fft
import json
import matplotlib.pyplot as plt

path = "20250709001.json"
lateral = "espina_base"
signal = "b"
comparingsignalA = "x"
comparingsignalB = "a"
indice_muestra = 1

patientdata = cargar_datos_json(path)['pruebas']
claves = list(patientdata.keys())  # Obtener las claves en orden

def filter_signal(signal, cutoff_hz=3, fs=50.0, order=20):
    nyquist = 0.5 * fs
    normal_cutoff = cutoff_hz / nyquist
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    filtered = filtfilt(b, a, signal)
    return filtered

def cargar_datos_json(ruta_archivo):
    """Carga datos desde un archivo JSON."""
    try:
        with open(ruta_archivo, 'r') as file:
            return json.load(file)
    except FileNotFoundError:
        print(f"Error: No se encontró el archivo '{ruta_archivo}'")
        return None
    except json.JSONDecodeError:
        print(f"Error: El archivo '{ruta_archivo}' no contiene un JSON válido")
        return None


def energy_spectrum(signal, fs, window_size_samples, step_size_samples, apply_window=True):

    energies = []
    times = []

    num_samples = len(signal)

    for start in range(0, num_samples - window_size_samples + 1, step_size_samples):
        end = start + window_size_samples
        window = signal[start:end]

        if apply_window:
            window = window * np.hanning(window_size_samples)

        energy = np.sum(window ** 2)
        energies.append(energy)

        t_center = (start + end) / 2 / fs  # tiempo en segundos
        times.append(t_center)

    energies = np.array(energies)
    times = np.array(times)

    # Normalizar la energía
    if np.max(np.abs(energies)) > 0:
        energies = energies / np.max(np.abs(energies))

    return times, energies

def binary_threshold(signal: np.ndarray, threshold: float = 0.2) -> np.ndarray:
    return (signal >= threshold).astype(int)

def promediomovil(senal, ventana=50, fs=50):
    senal = np.array(senal, dtype=float)
    kernel = np.ones(ventana) / ventana
    prom = np.convolve(senal, kernel, mode='valid')

    # Normalizar entre 0 y 1
    if np.max(prom) != 0:
        prom = prom / np.max(prom)

    # Tiempo alineado al final de cada ventana
    t = (np.arange(len(prom)) + ventana - 1) / fs
    return t, prom




def derivar_senal(senal, fs=50):
    senal = np.array(senal, dtype=float)
    derivada = np.diff(senal) * fs
    # Eje de tiempo centrado entre las muestras
    t_deriv = np.arange(len(derivada)) / fs + (0.5 / fs)
    return t_deriv, derivada




if 0 <= indice_muestra < len(claves):
    clave_muestra = claves[indice_muestra]
    sample = patientdata[clave_muestra][lateral]
    asignal = np.array([d[signal] for d in sample])
    acomparingsignalA = np.array([d[comparingsignalA] for d in sample])
    acomparingsignalB = np.array([d[comparingsignalB] for d in sample])

    # senal va a estar entre 0 y 1.
    asignal = asignal/max(abs(asignal))
    acomparingsignalA = acomparingsignalA/max(abs(acomparingsignalA))
    acomparingsignalB = acomparingsignalB/max(abs(acomparingsignalB))

    asignalfiltered = filter_signal(asignal)
    time = [d["t"] / 1000.0 for d in sample]

    plt.plot(time, asignal)
    plt.plot(time, acomparingsignalA)
    plt.plot(time, acomparingsignalB)

    plt.xlabel("Tiempo (s)")
    plt.ylabel(f"Señal {signal}")
    plt.title(f"Muestra #{indice_muestra}: {clave_muestra}")
    plt.grid(True)

    times, energies = energy_spectrum(
      signal=asignalfiltered,
      fs=50,
      window_size_samples=50,
      step_size_samples=1,
      apply_window=True
    )
    #plt.plot(times, energies)

    t_movil, pmovil = promediomovil(energies, ventana=50)

    t_derivada, derivada = derivar_senal(energies);
    binary = binary_threshold(pmovil, threshold=0.2)

    plt.plot(t_movil, binary)
    #plt.plot(t_movil, pmovil)


    plt.show()


## **Fases**

## **Levantamiento y sentado  - OK ✅**




In [ ]:
# Variables de control
path = "20250709004.json"
lateral = "espina_base"
signal = "b"
indice_muestra = 2  # Cambia este valor para seleccionar la muestra deseada

patientdata = cargar_datos_json(path)['pruebas']
claves = list(patientdata.keys())  # Obtener las claves en orden

if 0 <= indice_muestra < len(claves):
    clave_muestra = claves[indice_muestra]
    sample = patientdata[clave_muestra][lateral]
    asignal = np.array([d[signal] for d in sample])

    # senal va a estar entre 0 y 1.
    asignal = asignal/max(abs(asignal))
    asignalfiltered = filter_signal(asignal)
    time = [d["t"] / 1000.0 for d in sample]

    plt.plot(time, asignalfiltered)
    plt.xlabel("Tiempo (s)")
    plt.ylabel(f"Señal {signal}")
    plt.title(f"Muestra #{indice_muestra}: {clave_muestra}")
    plt.grid(True)

    times, energies = energy_spectrum(
      signal=asignalfiltered,
      fs=50,
      window_size_samples=50,
      step_size_samples=1,
      apply_window=True
    )
    plt.plot(times, energies)

    t_movil, pmovil = promediomovil(energies, ventana=50)

    #t_derivada, derivada = derivar_senal(energies);
    binary = binary_threshold(pmovil, threshold=0.2)

    plt.plot(t_movil, binary)
    plt.plot(t_movil, pmovil)


    plt.show()

else:
    print("Índice fuera de rango.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def detectar_fases_tug(tiempo, senal_binaria, duracion_minima=0.5):
    """
    Detecta las fases de levantamiento y sentado en la prueba TUG

    Args:
        tiempo: array con los tiempos
        senal_binaria: señal binaria (0 o 1)
        duracion_minima: duración mínima en segundos para considerar una fase válida

    Returns:
        dict con información de las fases detectadas
    """

    # Encontrar transiciones
    transiciones = np.diff(senal_binaria.astype(int))
    indices_subida = np.where(transiciones == 1)[0] + 1  # De 0 a 1
    indices_bajada = np.where(transiciones == -1)[0] + 1  # De 1 a 0

    # Si la señal empieza en 1, agregar el índice 0 como subida
    if senal_binaria[0] == 1:
        indices_subida = np.concatenate([[0], indices_subida])

    # Si la señal termina en 1, agregar el último índice como bajada
    if senal_binaria[-1] == 1:
        indices_bajada = np.concatenate([indices_bajada, [len(senal_binaria)-1]])

    # Emparejar subidas con bajadas
    fases_movimiento = []
    min_indices = min(len(indices_subida), len(indices_bajada))

    for i in range(min_indices):
        inicio_mov = indices_subida[i]
        fin_mov = indices_bajada[i]

        if inicio_mov < fin_mov:  # Verificar que la secuencia sea correcta
            t_inicio = tiempo[inicio_mov]
            t_fin = tiempo[fin_mov]
            duracion = t_fin - t_inicio

            # Solo considerar movimientos que superen la duración mínima
            if duracion >= duracion_minima:
                fases_movimiento.append({
                    'inicio_tiempo': t_inicio,
                    'fin_tiempo': t_fin,
                    'duracion': duracion,
                    'inicio_indice': inicio_mov,
                    'fin_indice': fin_mov
                })

    # Clasificar las fases
    fases_clasificadas = {
        'levantamiento': None,
        'sentado': None,
        'todas_las_fases': fases_movimiento
    }

    if len(fases_movimiento) >= 1:
        fases_clasificadas['levantamiento'] = fases_movimiento[0]

    if len(fases_movimiento) >= 2:
        fases_clasificadas['sentado'] = fases_movimiento[-1]  # La última fase

    return fases_clasificadas

def graficar_fases_tug(tiempo, senal_original, senal_binaria, t_movil, pmovil, fases):
    """
    Grafica las señales y marca las fases detectadas
    """
    plt.figure(figsize=(12, 8))

    # Gráfico principal
    plt.plot(tiempo, senal_original, label='Señal Original', alpha=0.7)
    plt.plot(t_movil, pmovil, label='Promedio Móvil', linewidth=2)
    plt.plot(t_movil, senal_binaria, label='Señal Binaria', linewidth=2)

    # Marcar las fases
    colores = ['red', 'orange', 'purple', 'brown']

    for i, fase in enumerate(fases['todas_las_fases']):
        color = colores[i % len(colores)]
        plt.axvspan(fase['inicio_tiempo'], fase['fin_tiempo'],
                   alpha=0.3, color=color,
                   label=f'Fase {i+1}: {fase["duracion"]:.2f}s')

    # Destacar levantamiento y sentado si están identificados
    if fases['levantamiento']:
        plt.axvspan(fases['levantamiento']['inicio_tiempo'],
                   fases['levantamiento']['fin_tiempo'],
                   alpha=0.5, color='green',
                   label=f'LEVANTAMIENTO: {fases["levantamiento"]["duracion"]:.2f}s')

    if fases['sentado']:
        plt.axvspan(fases['sentado']['inicio_tiempo'],
                   fases['sentado']['fin_tiempo'],
                   alpha=0.5, color='blue',
                   label=f'SENTADO: {fases["sentado"]["duracion"]:.2f}s')

    plt.xlabel("Tiempo (s)")
    plt.ylabel("Señal")
    plt.title(f"Detección de Fases TUG")
    plt.grid(True)
    plt.legend()
    plt.show()

def imprimir_resumen_fases(fases):
    """
    Imprime un resumen de las fases detectadas
    """
    print("\n" + "="*50)
    print("RESUMEN DE FASES DETECTADAS")
    print("="*50)

    print(f"Total de fases de movimiento detectadas: {len(fases['todas_las_fases'])}")

    if fases['levantamiento']:
        lev = fases['levantamiento']
        print(f"\n🚶 FASE DE LEVANTAMIENTO:")
        print(f"   Tiempo inicio: {lev['inicio_tiempo']:.2f} s")
        print(f"   Tiempo fin: {lev['fin_tiempo']:.2f} s")
        print(f"   Duración: {lev['duracion']:.2f} s")
    else:
        print("\n❌ No se detectó fase de levantamiento")

    if fases['sentado']:
        sent = fases['sentado']
        print(f"\n🪑 FASE DE SENTADO:")
        print(f"   Tiempo inicio: {sent['inicio_tiempo']:.2f} s")
        print(f"   Tiempo fin: {sent['fin_tiempo']:.2f} s")
        print(f"   Duración: {sent['duracion']:.2f} s")
    else:
        print("\n❌ No se detectó fase de sentado")

    print(f"\n📊 TODAS LAS FASES:")
    for i, fase in enumerate(fases['todas_las_fases']):
        print(f"   Fase {i+1}: {fase['inicio_tiempo']:.2f}s - {fase['fin_tiempo']:.2f}s (Duración: {fase['duracion']:.2f}s)")

# Tu código principal modificado
path = "20250709001.json"
lateral = "espina_base"
signal = "b"
indice_muestra = 3

patientdata = cargar_datos_json(path)['pruebas']
claves = list(patientdata.keys())

if 0 <= indice_muestra < len(claves):
    clave_muestra = claves[indice_muestra]
    sample = patientdata[clave_muestra][lateral]
    asignal = np.array([d[signal] for d in sample])

    # Senal va a estar entre 0 y 1.
    asignal = asignal/max(abs(asignal))
    asignalfiltered = filter_signal(asignal)
    time = [d["t"] / 1000.0 for d in sample]

    times, energies = energy_spectrum(
        signal=asignalfiltered,
        fs=50,
        window_size_samples=50,
        step_size_samples=1,
        apply_window=True
    )

    t_movil, pmovil = promediomovil(energies, ventana=50)
    binary = binary_threshold(pmovil, threshold=0.2)

    # NUEVA FUNCIONALIDAD: Detectar fases
    fases_detectadas = detectar_fases_tug(t_movil, binary, duracion_minima=0.5)

    # Mostrar resultados
    imprimir_resumen_fases(fases_detectadas)

    # Graficar con las fases marcadas
    graficar_fases_tug(time, asignalfiltered, binary, t_movil, pmovil, fases_detectadas)

else:
    print("Índice fuera de rango.")

## **Caminata - Ida / Vuelta**


In [ ]:
# Variables de control
path = "20250709008.json"
lateral = "izquierda"
signal = "g"
indice_muestra = 3  # Cambia este valor para seleccionar la muestra deseada

patientdata = cargar_datos_json(path)['pruebas']
claves = list(patientdata.keys())  # Obtener las claves en orden

if 0 <= indice_muestra < len(claves):
    clave_muestra = claves[indice_muestra]
    sample = patientdata[clave_muestra][lateral]
    asignal = np.array([d[signal] for d in sample])

    # senal va a estar entre 0 y 1.
    asignal = asignal/max(abs(asignal))
    asignalfiltered = filter_signal(asignal)
    time = [d["t"] / 1000.0 for d in sample]

    plt.plot(time, asignalfiltered)
    plt.xlabel("Tiempo (s)")
    plt.ylabel(f"Señal {signal}")
    plt.title(f"Muestra #{indice_muestra}: {clave_muestra}")
    plt.grid(True)

    times, energies = energy_spectrum(
      signal=asignalfiltered,
      fs=50,
      window_size_samples=50,
      step_size_samples=1,
      apply_window=True
    )
    plt.plot(times, energies)

    t_movil, pmovil = promediomovil(energies, ventana=50)

    #t_derivada, derivada = derivar_senal(energies);
    binary = binary_threshold(pmovil, threshold=0.2)

    plt.plot(t_movil, binary)
    plt.plot(t_movil, pmovil)


    plt.show()

else:
    print("Índice fuera de rango.")

## Señales cortadas para caminatas ida y vuelta

In [ ]:
def extraer_fase(time, signal, inicio_ms, fin_ms):
    """
    Filtra la señal y el tiempo para quedarse solo con el segmento
    comprendido entre inicio_ms y fin_ms (en milisegundos).
    """
    inicio_s = inicio_ms / 1000.0
    fin_s = fin_ms / 1000.0

    mask = (np.array(time) >= inicio_s) & (np.array(time) <= fin_s)

    return np.array(time)[mask], np.array(signal)[mask]


##Caminata E Ida

###Izquierda

In [ ]:
path = "20250709003.json"
path_fases = "20250709003.2.json"
lateral = "izquierda"
signal = "g"
indice_muestra = 1
fase_a_analizar = "caminataIda"         # Nombre de la fase

patientdata = cargar_datos_json(path)['pruebas']
claves = list(patientdata.keys())

with open(path_fases, "r") as f:
    fases_json = json.load(f)

if 0 <= indice_muestra < len(claves):
    clave_muestra = claves[indice_muestra]
    sample = patientdata[clave_muestra][lateral]

    # senal va a estar entre 0 y 1.
    asignal = np.array([d[signal] for d in sample])
    asignal = asignal / max(abs(asignal))
    asignalfiltered = filter_signal(asignal)
    time = np.array([d["t"] / 1000.0 for d in sample])

    # === Recortar la señal a la fase elegida ===
    inicio_ms = fases_json[fase_a_analizar]["inicio"]
    fin_ms = fases_json[fase_a_analizar]["fin"]

    time_fase, signal_fase = extraer_fase(time, asignalfiltered, inicio_ms, fin_ms)

    time_fase = time_fase - time_fase[0]

    # === Procesar solo la fase ===
    plt.plot(time_fase, signal_fase, label="Señal filtrada fase")

    times, energies = energy_spectrum(
        signal=signal_fase,
        fs=50,
        window_size_samples=50,
        step_size_samples=1,
        apply_window=True
    )
    plt.plot(times, energies, label="Energía")

    t_movil, pmovil = promediomovil(energies, ventana=50)

    #t_derivada, derivada = derivar_senal(energies);
    binary = binary_threshold(pmovil, threshold=0.2)

    plt.plot(t_movil, binary, label="Umbral binario")
    plt.plot(t_movil, pmovil, label="Promedio móvil")

    plt.xlabel("Tiempo (s)")
    plt.ylabel(f"Señal {signal}")
    plt.title(f"Muestra #{indice_muestra} - Fase: {fase_a_analizar}")
    plt.legend()
    plt.grid(True)
    plt.show()

else:
    print("Índice fuera de rango.")


###Derecha

In [ ]:
path = "20250709003.json"
path_fases = "20250709003.2.json"
lateral = "derecha"
signal = "g"
indice_muestra = 1
fase_a_analizar = "caminataIda"         # Nombre de la fase

patientdata = cargar_datos_json(path)['pruebas']
claves = list(patientdata.keys())

with open(path_fases, "r") as f:
    fases_json = json.load(f)

if 0 <= indice_muestra < len(claves):
    clave_muestra = claves[indice_muestra]
    sample = patientdata[clave_muestra][lateral]

    # senal va a estar entre 0 y 1.
    asignal = np.array([d[signal] for d in sample])
    asignal = asignal / max(abs(asignal))
    asignalfiltered = filter_signal(asignal)
    time = np.array([d["t"] / 1000.0 for d in sample])

    # === Recortar la señal a la fase elegida ===
    inicio_ms = fases_json[fase_a_analizar]["inicio"]
    fin_ms = fases_json[fase_a_analizar]["fin"]

    time_fase, signal_fase = extraer_fase(time, asignalfiltered, inicio_ms, fin_ms)

    time_fase = time_fase - time_fase[0]

    # === Procesar solo la fase ===
    plt.plot(time_fase, signal_fase, label="Señal filtrada fase")

    times, energies = energy_spectrum(
        signal=signal_fase,
        fs=50,
        window_size_samples=50,
        step_size_samples=1,
        apply_window=True
    )
    plt.plot(times, energies, label="Energía")

    t_movil, pmovil = promediomovil(energies, ventana=50)

    #t_derivada, derivada = derivar_senal(energies);
    binary = binary_threshold(pmovil, threshold=0.2)

    plt.plot(t_movil, binary, label="Umbral binario")
    plt.plot(t_movil, pmovil, label="Promedio móvil")

    plt.xlabel("Tiempo (s)")
    plt.ylabel(f"Señal {signal}")
    plt.title(f"Muestra #{indice_muestra} - Fase: {fase_a_analizar}")
    plt.legend()
    plt.grid(True)
    plt.show()

else:
    print("Índice fuera de rango.")


##CaminataVuelta

###Izquierda

In [ ]:
path = "20250709003.json"
path_fases = "20250709003.2.json"
lateral = "izquierda"
signal = "g"
indice_muestra = 1
fase_a_analizar = "caminataVuelta"         # Nombre de la fase

patientdata = cargar_datos_json(path)['pruebas']
claves = list(patientdata.keys())

with open(path_fases, "r") as f:
    fases_json = json.load(f)

if 0 <= indice_muestra < len(claves):
    clave_muestra = claves[indice_muestra]
    sample = patientdata[clave_muestra][lateral]

    # senal va a estar entre 0 y 1.
    asignal = np.array([d[signal] for d in sample])
    asignal = asignal / max(abs(asignal))
    asignalfiltered = filter_signal(asignal)
    time = np.array([d["t"] / 1000.0 for d in sample])

    # === Recortar la señal a la fase elegida ===
    inicio_ms = fases_json[fase_a_analizar]["inicio"]
    fin_ms = fases_json[fase_a_analizar]["fin"]

    time_fase, signal_fase = extraer_fase(time, asignalfiltered, inicio_ms, fin_ms)

    time_fase = time_fase - time_fase[0]

    # === Procesar solo la fase ===
    plt.plot(time_fase, signal_fase, label="Señal filtrada fase")

    times, energies = energy_spectrum(
        signal=signal_fase,
        fs=50,
        window_size_samples=50,
        step_size_samples=1,
        apply_window=True
    )
    plt.plot(times, energies, label="Energía")

    t_movil, pmovil = promediomovil(energies, ventana=50)

    #t_derivada, derivada = derivar_senal(energies);
    binary = binary_threshold(pmovil, threshold=0.2)

    plt.plot(t_movil, binary, label="Umbral binario")
    plt.plot(t_movil, pmovil, label="Promedio móvil")

    plt.xlabel("Tiempo (s)")
    plt.ylabel(f"Señal {signal}")
    plt.title(f"Muestra #{indice_muestra} - Fase: {fase_a_analizar}")
    plt.legend()
    plt.grid(True)
    plt.show()

else:
    print("Índice fuera de rango.")


###Derecha

In [ ]:
path = "20250709003.json"
path_fases = "20250709003.2.json"
lateral = "derecha"
signal = "g"
indice_muestra = 1
fase_a_analizar = "caminataVuelta"         # Nombre de la fase

patientdata = cargar_datos_json(path)['pruebas']
claves = list(patientdata.keys())

with open(path_fases, "r") as f:
    fases_json = json.load(f)

if 0 <= indice_muestra < len(claves):
    clave_muestra = claves[indice_muestra]
    sample = patientdata[clave_muestra][lateral]

    # senal va a estar entre 0 y 1.
    asignal = np.array([d[signal] for d in sample])
    asignal = asignal / max(abs(asignal))
    asignalfiltered = filter_signal(asignal)
    time = np.array([d["t"] / 1000.0 for d in sample])

    # === Recortar la señal a la fase elegida ===
    inicio_ms = fases_json[fase_a_analizar]["inicio"]
    fin_ms = fases_json[fase_a_analizar]["fin"]

    time_fase, signal_fase = extraer_fase(time, asignalfiltered, inicio_ms, fin_ms)

    time_fase = time_fase - time_fase[0]

    # === Procesar solo la fase ===
    plt.plot(time_fase, signal_fase, label="Señal filtrada fase")

    times, energies = energy_spectrum(
        signal=signal_fase,
        fs=50,
        window_size_samples=50,
        step_size_samples=1,
        apply_window=True
    )
    plt.plot(times, energies, label="Energía")

    t_movil, pmovil = promediomovil(energies, ventana=50)

    #t_derivada, derivada = derivar_senal(energies);
    binary = binary_threshold(pmovil, threshold=0.2)

    plt.plot(t_movil, binary, label="Umbral binario")
    plt.plot(t_movil, pmovil, label="Promedio móvil")

    plt.xlabel("Tiempo (s)")
    plt.ylabel(f"Señal {signal}")
    plt.title(f"Muestra #{indice_muestra} - Fase: {fase_a_analizar}")
    plt.legend()
    plt.grid(True)
    plt.show()

else:
    print("Índice fuera de rango.")


## **Giro - OK ✅**

In [ ]:
# Variables de control
path = "20250709003.json"
lateral = "espina_base"
signal = "g"
indice_muestra = 1  # Cambia este valor para seleccionar la muestra deseada

patientdata = cargar_datos_json(path)['pruebas']
claves = list(patientdata.keys())  # Obtener las claves en orden

if 0 <= indice_muestra < len(claves):
    clave_muestra = claves[indice_muestra]
    sample = patientdata[clave_muestra][lateral]
    asignal = np.array([d[signal] for d in sample])

    # senal va a estar entre 0 y 1.
    asignal = asignal/max(abs(asignal))
    asignalfiltered = filter_signal(asignal)
    time = [d["t"] / 1000.0 for d in sample]

    plt.plot(time, asignalfiltered)
    plt.xlabel("Tiempo (s)")
    plt.ylabel(f"Señal {signal}")
    plt.title(f"Muestra #{indice_muestra}: {clave_muestra}")
    plt.grid(True)

    times, energies = energy_spectrum(
      signal=asignalfiltered,
      fs=50,
      window_size_samples=50,
      step_size_samples=1,
      apply_window=True
    )
    plt.plot(times, energies)

    t_movil, pmovil = promediomovil(energies, ventana=50)

    #t_derivada, derivada = derivar_senal(energies);
    binary = binary_threshold(pmovil, threshold=0.2)

    plt.plot(t_movil, binary)
    plt.plot(t_movil, pmovil)


    plt.show()

else:
    print("Índice fuera de rango.")

## Señales cortadas para giro **REVISAR CON PROFE DOMICIANO**

In [ ]:
path = "20250709001.json"
path_fases = "20250709008.2.json"
lateral = "espina_base"
signal = "g"
indice_muestra = 1
fase_a_analizar = "giro"         # Nombre de la fase

patientdata = cargar_datos_json(path)['pruebas']
claves = list(patientdata.keys())

with open(path_fases, "r") as f:
    fases_json = json.load(f)

if 0 <= indice_muestra < len(claves):
    clave_muestra = claves[indice_muestra]
    sample = patientdata[clave_muestra][lateral]

    # senal va a estar entre 0 y 1.
    asignal = np.array([d[signal] for d in sample])
    asignal = asignal / max(abs(asignal))
    asignalfiltered = filter_signal(asignal)
    time = np.array([d["t"] / 1000.0 for d in sample])

    # === Recortar la señal a la fase elegida ===
    inicio_ms = fases_json[fase_a_analizar]["inicio"]
    fin_ms = fases_json[fase_a_analizar]["fin"]

    time_fase, signal_fase = extraer_fase(time, asignalfiltered, inicio_ms, fin_ms)

    time_fase = time_fase - time_fase[0]

    # === Procesar solo la fase ===
    plt.plot(time_fase, signal_fase, label="Señal filtrada fase")

    times, energies = energy_spectrum(
        signal=signal_fase,
        fs=50,
        window_size_samples=50,
        step_size_samples=1,
        apply_window=True
    )
    plt.plot(times, energies, label="Energía")

    t_movil, pmovil = promediomovil(energies, ventana=50)

    #t_derivada, derivada = derivar_senal(energies);
    binary = binary_threshold(pmovil, threshold=0.2)

    plt.plot(t_movil, binary, label="Umbral binario")
    plt.plot(t_movil, pmovil, label="Promedio móvil")

    plt.xlabel("Tiempo (s)")
    plt.ylabel(f"Señal {signal}")
    plt.title(f"Muestra #{indice_muestra} - Fase: {fase_a_analizar}")
    plt.legend()
    plt.grid(True)
    plt.show()

else:
    print("Índice fuera de rango.")


## Verificacion manual (SWING - STANCE)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, find_peaks

# =====================================================
# CARGAR DATOS
# =====================================================
path = "20250709003.json"
signal = "x"
indice_muestra = 1
fs=50.0

patientdata = cargar_datos_json(path)['pruebas']
claves = list(patientdata.keys())

if 0 <= indice_muestra < len(claves):
    clave_muestra = claves[indice_muestra]

    for lateral in ["izquierda", "derecha"]:
        sample = patientdata[clave_muestra][lateral]

        # Señal cruda y tiempo en segundos
        asignal = np.array([d[signal] for d in sample])
        time = np.array([d["t"] / 1000.0 for d in sample])

        # Normalización
        asignal = asignal / max(abs(asignal))

        # Filtrado (tu función personalizada)
        asignal_filtered = filter_signal(asignal,fs=fs)

        # Parámetros detección
        min_stride_s = 0.4
        distance = int(min_stride_s * fs)
        prominence = 0.2 * np.max(np.abs(asignal_filtered))

        # Picos y valles
        peaks, _ = find_peaks(asignal_filtered, prominence=prominence, distance=distance)
        valleys, _ = find_peaks(-asignal_filtered, prominence=prominence, distance=distance)

        # Picos y valles (sin filtro)
        # peaks, _ = find_peaks(asignal, prominence=prominence, distance=distance)
        # valleys, _ = find_peaks(-asignal, prominence=prominence, distance=distance)

        # Eventos y fases
        events = np.concatenate([
            np.stack([time[peaks], np.repeat("STANCE_START", len(peaks))], axis=1),
            np.stack([time[valleys], np.repeat("SWING_START", len(valleys))], axis=1)
        ])
        events = events[np.argsort(events[:, 0].astype(float))]

        fases = []
        for i in range(len(events) - 1):
            fase_actual = events[i][1]
            t_inicio = float(events[i][0])
            t_fin = float(events[i + 1][0])
            dur = t_fin - t_inicio
            if fase_actual == "STANCE_START":
                fases.append({"fase": "STANCE", "inicio": t_inicio, "fin": t_fin, "duracion": dur})
            elif fase_actual == "SWING_START":
                fases.append({"fase": "SWING", "inicio": t_inicio, "fin": t_fin, "duracion": dur})

        # ==============================
        # IMPRESIÓN DE RESULTADOS
        # ==============================
        print(f"\n==== {lateral.upper()} ====")
        print("fase\t| inicio(s)\t| fin(s)\t| duracion(s)")
        for f in fases:
            print(f"{f['fase']}\t| {f['inicio']:.2f}\t| {f['fin']:.2f}\t| {f['duracion']:.2f}")

        mean_stance = np.mean([f['duracion'] for f in fases if f['fase'] == 'STANCE']) if any(f['fase']=='STANCE' for f in fases) else 0
        mean_swing = np.mean([f['duracion'] for f in fases if f['fase'] == 'SWING']) if any(f['fase']=='SWING' for f in fases) else 0
        print(f"\nPromedios - {lateral}: STANCE={mean_stance:.2f}s, SWING={mean_swing:.2f}s")

        # ==============================
        # GRÁFICA
        # ==============================
        plt.figure(figsize=(16,6))
        plt.plot(time, asignal_filtered, label=f"{lateral}")
        plt.plot(time[peaks], asignal_filtered[peaks], "ro", label="Picos (STANCE)")
        plt.plot(time[valleys], asignal_filtered[valleys], "go", label="Valles (SWING)")

        # plt.figure(figsize=(16,6))
        # plt.plot(time, asignal, label=f"{lateral}")
        # plt.plot(time[peaks], asignal[peaks], "ro", label="Picos (STANCE)")
        # plt.plot(time[valleys], asignal[valleys], "go", label="Valles (SWING)")

        for f in fases:
            color = "red" if f["fase"] == "STANCE" else "green"
            plt.axvspan(f["inicio"], f["fin"], color=color, alpha=0.25, label=f["fase"] if f == fases[0] else "")

        # --- Mostrar TODO el rango temporal en segundos ---
        total_time = time[-1] - time[0]
        plt.xlim(time[0], time[-1])  # muestra toda la caminata completa

        # --- Forzar ticks del eje X cada 1 segundo ---
        num_seconds = int(np.ceil(time[-1]))
        plt.xticks(np.arange(0, num_seconds + 1, 1.0))

        plt.xlabel("Tiempo (s)")
        plt.ylabel(f"Señal {signal}")
        plt.title(f"{lateral.capitalize()} - Fases STANCE / SWING (Filtro Butterworth)")
        plt.grid(True, which="both", axis="both", linestyle="--", alpha=0.5)
        plt.legend()
        plt.tight_layout()
        plt.show()
else:
    print("Índice fuera de rango.")


### ¿Que es la autocorrelacion?

La autocorrelación es básicamente medir cuán parecida es una señal con una versión desplazada de sí misma en el tiempo. Es como superponer la señal contigo misma, pero desplazada unos instantes, y ver cuánto coinciden. Es decir, mide la similitud de la señal con ella misma después de un cierto retraso o lag.

### ¿Para que sirve la autocorrelacion?

* Al aplicar autocorrelación a la señal (por ejemplo, del eje y de un acelerómetro en el tobillo), las puntas de la autocorrelación indican periodos repetitivos. Por ejemplo, si la persona da pasos cada ~1 segundo, habrá un pico en la autocorrelación alrededor de ese lag. Esto te dice que hay ciclos repetitivos de esa duración.

* En pruebas como la TUG, la marcha tiene un carácter semi-periódico: ciclos de paso claros durante caminar, pero interrumpidos en giros o sentarse. La autocorrelación te permite:
  * Estimar el período promedio entre pasos.
  * Identificar cuándo el comportamiento deja de ser cíclico (por ejemplo, durante giros o microsaltos).
  * Probar si la señal es cíclica (regularidad en la marcha) o tiene irregularidades.

## Swing - Stance

### *CORTES EN EL TIEMPO PARA LA FASE DE GIRO*

In [ ]:
# Parámetros
path = "20250709001.json"
lateral = "espina_base"
signal = "a"
indice_muestra = 3

# Cargar datos
patientdata = cargar_datos_json(path)['pruebas']
claves = list(patientdata.keys())

if 0 <= indice_muestra < len(claves):
    clave_muestra = claves[indice_muestra]
    sample = patientdata[clave_muestra][lateral]

    # Señal cruda normalizada
    asignal = np.array([d[signal] for d in sample], dtype=float)
    asignal = asignal / np.max(np.abs(asignal))
    time = np.array([d["t"] / 1000.0 for d in sample], dtype=float)

    # Filtrar señal
    asignalfiltered = filter_signal(asignal)

    # Calcular energía y promedio móvil
    times, energies = energy_spectrum(asignalfiltered, fs=50, window_size_samples=50, step_size_samples=1)
    t_movil, pmovil = promediomovil(energies, ventana=50)

    # Umbral binario
    binary = binary_threshold(pmovil, threshold=0.2)

    # Graficar señal y umbral binario
    plt.figure(figsize=(10,5))
    plt.plot(time, asignalfiltered, label="Señal filtrada (a)")
    plt.plot(t_movil, pmovil, label="Promedio móvil", color="orange")
    plt.plot(t_movil, binary, label="Binary threshold", color="green")
    plt.xlabel("Tiempo (s)")
    plt.ylabel("Amplitud / Energía")
    plt.title(f"Detección de giros - {clave_muestra}")
    plt.grid(True)
    plt.legend()
    plt.xticks(np.arange(0, max(time)+1, 1))
    plt.show()

    # ---- DETECCIÓN DE CORTES ----
    cambios = np.diff(binary)
    inicios = np.where(cambios == 1)[0]
    fines = np.where(cambios == -1)[0]

    # Ajuste: asegurar que los índices coincidan en pares
    if len(fines) < len(inicios):
        fines = np.append(fines, len(binary) - 1)

    # Mostrar intervalos
    print("🌀 Intervalos de giro detectados:")
    for i, inicio in enumerate(inicios):
        fin = fines[i] if i < len(fines) else None
        duracion = t_movil[fin] - t_movil[inicio]
        print(f"Giro {i+1}: inicio = {t_movil[inicio]:.2f}s, fin = {t_movil[fin]:.2f}s, duración = {duracion:.2f}s")

else:
    print("Índice fuera de rango.")

# ---- DETECCIÓN DE CORTES ----
cambios = np.diff(binary)
inicios = np.where(cambios == 1)[0]
fines   = np.where(cambios == -1)[0]

# Ajustar pares
if len(fines) < len(inicios):
    fines = np.append(fines, len(binary) - 1)

# Guardar intervalos en lista estructurada
intervalos_giro = []

for i, inicio in enumerate(inicios):
    fin = fines[i] if i < len(fines) else None
    if fin is None:
        continue
    intervalos_giro.append({
        "inicio": float(t_movil[inicio]),
        "fin": float(t_movil[fin]),
        "duracion": float(t_movil[fin] - t_movil[inicio])
    })

# Imprimirlos (opcional)
print("🌀 Intervalos de giro detectados:")
for i, g in enumerate(intervalos_giro):
    print(f"Giro {i+1}: inicio={g['inicio']:.2f}s, fin={g['fin']:.2f}s, dur={g['duracion']:.2f}s")


## Caracterizacion de endpoints "IMPORTANTES" en Vimov para Integración con TUG

## Configruración Base
```javascript
API_ENDPOINT = "http://localhost:5000/api" // Desarrollo
API_ENDPOINT = "https://i2thub.icesi.edu.co:5443/pdvimov/api" // Produccion
```

## 1. getPatientsApi

**Proposito:** Obtener una lista paginada de pacientes con filtros de bÃºsqueda

**Endpoint:** `GET /api/patients`

**Funcion:** `getPatientsApi(filter, position, take, token)`

**Ubicacion:** `src/pdaily/api/index.js` lineas 55-75

---

## 2. getAllPatientsApi

**Proposito:** Obtener la lista completa de todos los pacientes sin paginacion

**Endpoint:** `GET /api/patients/all`

**Funcion:** `getAllPatientsApi(token)`

**Ubicacion:** `src/pdaily/api/index.js` lineas 77-91

---

## 3. getPatientTestsApi

**Proposito:** Obtener los tests de movimiento de un paciente especifico por tipo de articulacion (ankle, arms)

**Endpoint:** `GET /api/patients/{patientId}/tests/{joint}`

**Funcion:** `getPatientTestsApi(patientId, joint, limit, offset, token)`

**Ubicacion:** `src/pdaily/api/index.js` lineas 123-136

---

## 4. getTestByIdApi

**Proposito:** Obtener los datos completos de un test especifico usando su ID

**Endpoint:** `GET /api/tests/{testId}`

**Funcion:** `getTestByIdApi(testId, token)`

**Ubicacion:** `src/pdaily/api/index.js` lineas 138-147

---

## 5. getPatientByIdApi

**Proposito:** Obtener la informaciÃ³n completa de un paciente especifico usando su ID

**Endpoint:** `GET /api/patients/{patientId}`

**Funcion:** `getPatientByIdApi(patientId, token)`

**Ubicacion:** `src/pdaily/api/index.js` lineas 297-306

---

## Headers Requeridos
Todos los endpoints requieren:
```javascript
headers: {
  'Authorization': `Bearer ${token}`,
  'Content-Type': 'application/json'
}
```

## **Función para conteo de pasos**

In [ ]:
# =========================================================
# IMPORTACIONES
# =========================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, butter, filtfilt
import json

# =========================================================
# FUNCIÓN DE FILTRADO
# =========================================================
def filter_signal(signal, cutoff_hz=3, fs=50.0, order=20):
    """
    Aplica un filtro pasa-bajo Butterworth a la señal.
    """
    nyquist = 0.5 * fs
    normal_cutoff = cutoff_hz / nyquist
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return filtfilt(b, a, signal)

# =========================================================
# FUNCIÓN PARA CARGAR DATOS DEL JSON
# =========================================================
def cargar_datos_json(path):
    with open(path, "r") as f:
        return json.load(f)

# =========================================================
# VARIABLES DE CONTROL
# =========================================================
path = "20250709001.json"
signal = "y"           # componente del sensor que analizas
indice_muestra = 1      # muestra a analizar

# Cargar datos
patientdata = cargar_datos_json(path)['pruebas']
claves = list(patientdata.keys())

# =========================================================
# PROCESAMIENTO DE DATOS
# =========================================================
if 0 <= indice_muestra < len(claves):
    clave_muestra = claves[indice_muestra]

    for lateral in ["izquierda", "derecha"]:
        sample = patientdata[clave_muestra][lateral]

        # Señal cruda y tiempo en segundos
        asignal = np.array([d[signal] for d in sample])
        time = np.array([d["t"] / 1000.0 for d in sample])

        # Normalización
        asignal = asignal / max(abs(asignal))

        # Filtrado
        asignalfiltered = filter_signal(asignal)

        # =========================================================
        # PRIMERA GRÁFICA (solo señal)
        # =========================================================
        plt.figure(figsize=(10, 4))
        plt.plot(time, asignalfiltered, label=f"{lateral}")
        plt.xlabel("Tiempo (s)")
        plt.ylabel(f"Señal {signal}")
        plt.title(f"{lateral.capitalize()} - Señal sin detección")
        plt.grid(True)
        plt.legend()
        plt.show()

        # =========================================================
        # SEGUNDA GRÁFICA (detección SWING/STANCE)
        # =========================================================
        dt = np.median(np.diff(time))
        fs = 1.0 / dt if dt > 0 else 50.0

        # Parámetros de detección adaptados a 50 Hz
        min_stride_s = 1.0  # duración mínima de paso
        distance = int(min_stride_s * fs)

        # Umbrales relativos
        height = 0.35 * np.max(np.abs(asignalfiltered))
        prominence = 0.35 * np.max(np.abs(asignalfiltered))

        # Detección de picos (STANCE) y valles (SWING)
        peaks, props_p = find_peaks(asignalfiltered, prominence=prominence, distance=distance)
        valleys, props_v = find_peaks(-asignalfiltered, prominence=prominence, distance=distance)

        # --- Graficar ---
        plt.figure(figsize=(12, 5))
        plt.plot(time, asignalfiltered, label=f"{lateral}")

        # Marcar picos (HEEL STRIKE → STANCE)
        plt.plot(time[peaks], asignalfiltered[peaks], "ro", label="Pico (STANCE)")
        # Marcar valles (TOE OFF → SWING)
        plt.plot(time[valleys], asignalfiltered[valleys], "go", label="Valle (SWING)")

        # Colorear regiones SWING/STANCE
        for i in range(len(peaks) - 1):
            t1, t2 = time[peaks[i]], time[peaks[i + 1]]
            plt.axvspan(t1, t2, color="orange", alpha=0.2, label="SWING" if i == 0 else "")
        for i in range(len(valleys) - 1):
            t1, t2 = time[valleys[i]], time[valleys[i + 1]]
            plt.axvspan(t1, t2, color="blue", alpha=0.1, label="STANCE" if i == 0 else "")

        plt.xlabel("Tiempo (s)")
        plt.ylabel(f"Señal {signal}")
        plt.title(f"{lateral.capitalize()} - Detección SWING/STANCE")
        plt.grid(True)
        plt.legend()
        plt.show()

        # =========================================================
        # IMPRESIÓN DE RESULTADOS
        # =========================================================
        print("-------------------------------------------------")
        print(f"Lado: {lateral}")
        print("DATOS PEAKS (STANCE)")
        print("Indices:", peaks)
        print("Prominencia:", props_p.get("prominences"))
        print("Left bases:", props_p.get("left_bases"))
        print("Right bases:", props_p.get("right_bases"))

        print("\nDATOS VALLEYS (SWING)")
        print("Indices:", valleys)
        print("Prominencia:", props_v.get("prominences"))
        print("Left bases:", props_v.get("left_bases"))
        print("Right bases:", props_v.get("right_bases"))

        # Tiempo total entre picos y valles
        if len(peaks) > 1:
            print("\nTiempo total (peaks):", time[peaks[-1]] - time[peaks[0]])
        if len(valleys) > 1:
            print("Tiempo total (valleys):", time[valleys[-1]] - time[valleys[0]])
        print("-------------------------------------------------\n")
else:
    print("Índice fuera de rango.")


## JOANDGOAG

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

# ============================================
# 1. OBTENER INTERVALOS: LEVANTAMIENTO, GIRO1, GIRO2
# ============================================

# --- Fases TUG desde tu algoritmo ---
fases = fases_detectadas                 # levantamiento y sentado
giros = intervalos_giro                  # lista con GIRO1 y GIRO2

t_lev_fin   = fases["levantamiento"]["fin_tiempo"]
t_giro1_ini = giros[0]["inicio"]
t_giro1_fin = giros[0]["fin"]
t_giro2_ini = giros[1]["inicio"]
t_giro2_fin = giros[1]["fin"]

# --- DEFINIR CAMINATAS ---
t_inicio_ida    = t_lev_fin
t_fin_ida       = t_giro1_ini

t_inicio_vuelta = t_giro1_fin
t_fin_vuelta    = t_giro2_ini


# =====================================================
# 2. FUNCIÓN PARA DETECTAR SWING/STANCE EN INTERVALOS
# =====================================================
def detectar_pasos_intervalo(time, asignal_filtered, fs, t_inicio, t_fin):

    # Recorte por tiempo
    mask = (time >= t_inicio) & (time <= t_fin)

    t_seg = time[mask]
    sig_seg = asignal_filtered[mask]

    if len(sig_seg) < 10:
        return [], [], [], [], []

    # Parámetros
    min_stride_s = 0.4
    distance = int(min_stride_s * fs)
    prominence = 0.25 * np.max(np.abs(sig_seg))

    # Detectar picos (STANCE)
    peaks, _ = find_peaks(sig_seg, prominence=prominence, distance=distance)

    # Detectar valles (SWING)
    valleys, _ = find_peaks(-sig_seg, prominence=prominence, distance=distance)

    # Construir lista de eventos ordenados
    eventos = np.concatenate([
        np.stack([t_seg[peaks], np.repeat("STANCE_START", len(peaks))], axis=1),
        np.stack([t_seg[valleys], np.repeat("SWING_START", len(valleys))], axis=1)
    ])

    eventos = eventos[np.argsort(eventos[:, 0].astype(float))]

    fases_det = []
    for i in range(len(eventos) - 1):
        fase = eventos[i][1]
        t_i = float(eventos[i][0])
        t_f = float(eventos[i + 1][0])
        dur = t_f - t_i

        if fase == "STANCE_START":
            fases_det.append({"fase": "STANCE", "inicio": t_i, "fin": t_f, "duracion": dur})
        else:
            fases_det.append({"fase": "SWING", "inicio": t_i, "fin": t_f, "duracion": dur})

    return fases_det, t_seg, sig_seg, peaks, valleys


# =====================================================
# 3. PROCESAR AMBOS TOBILLOS SOLO EN CAMINATAS
# =====================================================
path = "20250709003.json"
signal = "x"
indice_muestra = 1
fs = 50

patientdata = cargar_datos_json(path)['pruebas']
clave = list(patientdata.keys())[indice_muestra]

for lateral in ["izquierda", "derecha"]:

    sample = patientdata[clave][lateral]

    asignal = np.array([d[signal] for d in sample])
    time = np.array([d["t"]/1000 for d in sample])

    asignal = asignal / np.max(np.abs(asignal))
    asignal_filtered = filter_signal(asignal, fs=fs)

    print("\n===================================================")
    print(f"   >>>>> {lateral.upper()} — CAMINATA DE IDA <<<<<")
    print("===================================================")

    fases_ida, t_seg_ida, sig_seg_ida, peaks_ida, valleys_ida = detectar_pasos_intervalo(
        time, asignal_filtered, fs, t_inicio_ida, t_fin_ida
    )

    print("fase\t| inicio\t| fin\t| duracion")
    for f in fases_ida:
        print(f"{f['fase']}\t| {f['inicio']:.2f}\t| {f['fin']:.2f}\t| {f['duracion']:.2f}")

    # PROMEDIOS
    stance_mean = (
      np.mean([x["duracion"] for x in fases_ida if x["fase"] == "STANCE"])
      if any(x["fase"] == "STANCE" for x in fases_ida)
      else 0
    )

    swing_mean = (
      np.mean([x["duracion"] for x in fases_ida if x["fase"] == "SWING"])
      if any(x["fase"] == "SWING" for x in fases_ida)
      else 0
    )

    print(f"\nPromedios (IDA): STANCE={stance_mean:.2f}s  SWING={swing_mean:.2f}s")


    # ===== VUELTA =====
    print("\n===================================================")
    print(f"   >>>>> {lateral.upper()} — CAMINATA DE VUELTA <<<<<")
    print("===================================================")

    fases_vuelta, t_seg_vuelta, sig_seg_vuelta, peaks_vuelta, valleys_vuelta = detectar_pasos_intervalo(
        time, asignal_filtered, fs, t_inicio_vuelta, t_fin_vuelta
    )

    print("fase\t| inicio\t| fin\t| duracion")
    for f in fases_vuelta:
        print(f"{f['fase']}\t| {f['inicio']:.2f}\t| {f['fin']:.2f}\t| {f['duracion']:.2f}")

    stance_mean_v = (
      np.mean([x["duracion"] for x in fases_vuelta if x["fase"] == "STANCE"])
      if any(x["fase"] == "STANCE" for x in fases_vuelta)
      else 0
    )

    swing_mean_v = (
      np.mean([x["duracion"] for x in fases_vuelta if x["fase"] == "SWING"])
      if any(x["fase"] == "SWING" for x in fases_vuelta)
      else 0
    )

    print(f"\nPromedios (VUELTA): STANCE={stance_mean:.2f}s  SWING={swing_mean:.2f}s")


    # ===========================================
    # 4. GRAFICAR SOLO LA CAMINATA DE IDA Y VUELTA
    # ===========================================
    plt.figure(figsize=(16,6))
    plt.plot(time, asignal_filtered, label=f"{lateral}")

    # IDA
    plt.plot(t_seg_ida[peaks_ida], sig_seg_ida[peaks_ida], "ro", label="STANCE (Ida)")
    plt.plot(t_seg_ida[valleys_ida], sig_seg_ida[valleys_ida], "go", label="SWING (Ida)")
    plt.axvspan(t_inicio_ida, t_fin_ida, color="blue", alpha=0.15, label="Caminata IDA")

    # VUELTA
    plt.plot(t_seg_vuelta[peaks_vuelta], sig_seg_vuelta[peaks_vuelta], "r^", label="STANCE (Vuelta)")
    plt.plot(t_seg_vuelta[valleys_vuelta], sig_seg_vuelta[valleys_vuelta], "g^", label="SWING (Vuelta)")
    plt.axvspan(t_inicio_vuelta, t_fin_vuelta, color="green", alpha=0.15, label="Caminata VUELTA")

    plt.xlabel("Tiempo (s)")
    plt.ylabel("Señal filtrada")
    plt.title(f"{lateral.upper()} - Detección de SWING/STANCE SOLO EN CAMINATAS")
    plt.grid(True)
    plt.legend()
    plt.show()


# Codigo PDG Actual.

## Descarte de datos

### Cargar Datos

In [ ]:
import os
import glob
import json
import numpy as np
import pandas as pd
from google.colab import data_table
from google.colab import drive

drive.mount('/content/drive')

# 1. Configuración de la ruta de Google Drive para evitar pérdidas de datos al reiniciar
DRIVE_TUG_PATH = "/content/drive/MyDrive/TUG_Data/"

def buscar_archivos_tug(folder_path=DRIVE_TUG_PATH):
    """
    Busca y retorna la lista de archivos JSON disponibles en la carpeta de Drive.
    """
    if not os.path.exists(folder_path):
        print(f"Advertencia: No se encontró la ruta {folder_path}. Asegúrate de montar el Drive correctamente.")
        return []

    archivos = glob.glob(os.path.join(folder_path, "*.json"))
    print(f"Se encontraron {len(archivos)} archivos JSON en Google Drive.")
    return sorted(archivos)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Ajustar y Mapear datos

In [ ]:
import os
import numpy as np

# ---------------------------------------------------------------------------
# Mapa deviceId -> clave del pipeline. Gana la primera coincidencia (en minúsculas).
# Funciona también con el prefijo "K1-" (K1-RIGHT-ANKLE, etc.).
# Si en tu pipeline viejo derecha/izquierda eran las MANOS, cambia esas dos líneas.
# ---------------------------------------------------------------------------
MAPA_SENSORES = [
    ("spine",       "espina_base"),
    ("base",        "espina_base"),
    ("right-ankle", "derecha"),
    ("left-ankle",  "izquierda"),
    ("right-hand",  "mano_derecha"),
    ("left-hand",   "mano_izquierda"),
]

# Escalas FIJAS por canal (antes se decidía muestra por muestra y rompía la señal).
# Acelerómetro del JSON nuevo: g.  Giróscopo del JSON nuevo: °/s.
ESCALA_ACC = 4096.0
ESCALA_GYRO = 1000.0


def _clave_sensor(device_id):
    d = str(device_id).lower()
    for patron, clave in MAPA_SENSORES:
        if patron in d:
            return clave
    return None  # dispositivo desconocido: se ignora


def adaptar_esquema_json(json_data, fs_destino=50.0):
    """
    Convierte el formato nuevo (Vimov, 'imuData') al formato clásico
    {'pruebas': {timestamp: {sensor: [registros]}}}, remuestreando cada sensor
    a fs_destino Hz por interpolación lineal.
    """
    if "pruebas" in json_data:
        return json_data
    if "imuData" not in json_data:
        return None

    timestamp_clave = str(json_data.get("timestamp", json_data.get("endTimestamp", "2026-01-01T00:00:00")))

    # 1. Agrupar por sensor físico
    grupos = {}
    for reg in json_data["imuData"]:
        clave = _clave_sensor(reg.get("deviceId", ""))
        if clave is not None:
            grupos.setdefault(clave, []).append(reg)

    estructurado = {}
    for clave, registros in grupos.items():
        registros = sorted(registros, key=lambda r: r.get("timestamp", 0))
        if len(registros) < 2:
            continue

        # Unidad del timestamp: ms en estos archivos; si fuera ns (>1e15) se pasa a ms
        primer_ts = registros[0].get("timestamp", 0)
        factor = 1e6 if primer_ts > 1e15 else 1.0
        t_cero = primer_ts / factor

        ts = np.array([r.get("timestamp", 0) / factor for r in registros], dtype=float)
        t_orig = (ts - t_cero) / 1000.0  # segundos relativos al primer dato del sensor

        def canal(grupo, eje):
            return np.array([r.get(grupo, {}).get(eje, 0.0) for r in registros], dtype=float)

        duracion = t_orig[-1] - t_orig[0]
        n = int(np.floor(duracion * fs_destino)) + 1
        t_dest = np.arange(n) / fs_destino

        acc_x = np.interp(t_dest, t_orig, canal("accelerometer", "x")) * ESCALA_ACC
        acc_y = np.interp(t_dest, t_orig, canal("accelerometer", "y")) * ESCALA_ACC
        acc_z = np.interp(t_dest, t_orig, canal("accelerometer", "z")) * ESCALA_ACC
        gyr_x = np.interp(t_dest, t_orig, canal("gyroscope", "x")) * ESCALA_GYRO
        gyr_y = np.interp(t_dest, t_orig, canal("gyroscope", "y")) * ESCALA_GYRO
        gyr_z = np.interp(t_dest, t_orig, canal("gyroscope", "z")) * ESCALA_GYRO

        estructurado[clave] = [
            {
                "t": float(t_dest[i] * 1000.0),
                "x": int(acc_x[i]), "y": int(acc_y[i]), "z": int(acc_z[i]),
                "a": int(gyr_x[i]), "b": int(gyr_y[i]), "g": int(gyr_z[i]),
                "millis": int(t_cero + t_dest[i] * 1000.0),
            }
            for i in range(n)
        ]

    return {
        "pruebas": {timestamp_clave: estructurado},
        # Metadatos opcionales (bórralos si tu validador no los necesita)
        "meta": {
            "testType": json_data.get("testType"),
            "hasWarning": json_data.get("hasWarning", False),
            "warningReason": json_data.get("warningReason"),
            "pausedDurationMs": json_data.get("pausedDurationMs", 0),
        },
    }


# ---------------------------------------------------------------------------
# Helpers para las gráficas
# ---------------------------------------------------------------------------
def cargar_prueba_adaptada(nombre_o_ruta, fs_destino=50.0):
    """Carga un JSON (nuevo o clásico) y devuelve {clave_prueba: {sensor: [registros]}}."""
    ruta = nombre_o_ruta
    if not os.path.isabs(ruta):
        ruta = os.path.join(DRIVE_TUG_PATH, ruta)

    crudo = cargar_datos_json_local(ruta)
    if not crudo:
        raise ValueError(f"Archivo vacío o JSON corrupto: {ruta}")

    adaptado = adaptar_esquema_json(crudo, fs_destino=fs_destino)
    if not adaptado or "pruebas" not in adaptado:
        raise ValueError(f"Formato no reconocido: {ruta}")
    return adaptado["pruebas"]


def extraer_senal(pruebas, indice_muestra, lateral, signal):
    """Devuelve (clave_muestra, tiempo_en_s, senal_normalizada)."""
    claves = list(pruebas.keys())
    if not (0 <= indice_muestra < len(claves)):
        raise IndexError(f"Índice {indice_muestra} fuera de rango (hay {len(claves)} muestra(s)).")

    clave = claves[indice_muestra]
    if lateral not in pruebas[clave]:
        raise KeyError(f"El sensor '{lateral}' no existe en esta prueba. Disponibles: {list(pruebas[clave].keys())}")

    sample = pruebas[clave][lateral]
    if not sample:
        raise ValueError(f"El sensor '{lateral}' no tiene datos.")

    asignal = np.array([d[signal] for d in sample], dtype=float)
    maximo = np.max(np.abs(asignal))
    if maximo > 0:
        asignal = asignal / maximo
    time = np.array([d["t"] / 1000.0 for d in sample], dtype=float)
    return clave, time, asignal

### Validación y Filtración

In [ ]:
def validar_y_filtrar_prueba(json_data, nombre_archivo="", fs_esperada=50.0, tolerancia_fs=5.0, duracion_minima_s=5.0):
    """
    Realiza un análisis de calidad de los datos de la prueba TUG para decidir si se acepta o descarta.

    Criterios de descarte:
    - Falta de estructura básica ('pruebas').
    - Inexistencia de los sensores requeridos ('izquierda', 'derecha', 'espina_base').
    - Frecuencia de muestreo promedio fuera del rango tolerable.
    - Señales vacías o duración total extremadamente corta.
    - Presencia de datos planos/congelados (varianza cero) o valores corruptos.
    """
    informe = {
        "archivo": os.path.basename(nombre_archivo),
        "valido": True,
        "motivo_descarte": [],
        "detalles": {}
    }

    if "pruebas" not in json_data:
        informe["valido"] = False
        informe["motivo_descarte"].append("No contiene la clave raíz 'pruebas'")
        return informe

    pruebas = json_data["pruebas"]
    if not pruebas:
        informe["valido"] = False
        informe["motivo_descarte"].append("El diccionario de pruebas está vacío")
        return informe

    # Evaluamos el primer timestamp disponible
    timestamp = list(pruebas.keys())[0]
    datos_timestamp = pruebas[timestamp]

    sensores_requeridos = ["izquierda", "derecha", "espina_base"]
    for sensor in sensores_requeridos:
        if sensor not in datos_timestamp:
            informe["valido"] = False
            informe["motivo_descarte"].append(f"Falta el sensor crítico: '{sensor}'")
            return informe

    # Validación detallada de señales por sensor
    for sensor in sensores_requeridos:
        registros = datos_timestamp[sensor]
        if len(registros) < 10:
            informe["valido"] = False
            informe["motivo_descarte"].append(f"Sensor '{sensor}' contiene muy pocas muestras ({len(registros)})")
            return informe

        # Extraer tiempos y comprobar fs
        tiempos = np.array([r["t"] for r in registros if "t" in r], dtype=float) / 1000.0 # en segundos
        if len(tiempos) < 2:
            informe["valido"] = False
            informe["motivo_descarte"].append(f"Sensor '{sensor}' no tiene un vector de tiempo válido")
            return informe

        duracion = tiempos[-1] - tiempos[0]
        if duracion < duracion_minima_s:
            informe["valido"] = False
            informe["motivo_descarte"].append(f"Duración insuficiente en '{sensor}': {duracion:.2f}s (Mínimo: {duracion_minima_s}s)")

        # Calcular frecuencia de muestreo real
        dts = np.diff(tiempos)
        dts = dts[dts > 0] # Evitar divisiones por cero o muestras repetidas simultáneas
        if len(dts) > 0:
            fs_real = 1.0 / np.mean(dts)
        else:
            fs_real = 0.0

        if abs(fs_real - fs_esperada) > tolerancia_fs:
            informe["valido"] = False
            informe["motivo_descarte"].append(f"Frecuencia de muestreo anómala en '{sensor}': {fs_real:.2f} Hz (Esperada: {fs_esperada} Hz)")

        # Verificar consistencia de ejes para evitar señales corruptas/congeladas
        for eje in ["x", "y", "z", "a", "b", "g"]:
            valores = np.array([r[eje] for r in registros if eje in r], dtype=float)
            if len(valores) == 0:
                informe["valido"] = False
                informe["motivo_descarte"].append(f"Falta el eje '{eje}' en el sensor '{sensor}'")
            elif np.var(valores) == 0:
                informe["valido"] = False
                informe["motivo_descarte"].append(f"Señal congelada (varianza cero) detectada en {sensor}.{eje}")

        informe["detalles"][sensor] = {
            "muestras": len(registros),
            "duracion_s": float(duracion),
            "fs_promedio": float(fs_real)
        }

    return informe

### Resultado

In [ ]:
procesar_n_muestras = 78

# Obtener lista completa de archivos en Drive
archivos_procesar = buscar_archivos_tug()

reporte_final = []

for ruta in archivos_procesar:
    nombre_archivo = os.path.basename(ruta)
    try:
        datos_crudos = cargar_datos_json(ruta)
        if not datos_crudos:
            reporte_final.append({
                "Archivo": nombre_archivo,
                "Estado": "❌ ERROR DE CARGA",
                "Detalle": "Archivo vacío o JSON corrupto",
                "Muestras": 0
            })
            continue

        # Aplicar adaptador dinámico de formato y remuestreo
        datos_adaptados = adaptar_esquema_json(datos_crudos, fs_destino=50.0)

        # Evaluar calidad
        calidad = validar_y_filtrar_prueba(datos_adaptados, nombre_archivo=ruta)

        if calidad["valido"]:
            # Extraemos info de muestras del primer sensor que encontremos para reportar
            sensor_temp = list(calidad["detalles"].keys())[0]
            muestras_interp = calidad["detalles"][sensor_temp]["muestras"]

            reporte_final.append({
                "Archivo": nombre_archivo,
                "Estado": "✅ ACEPTADO",
                "Detalle": "Formato normalizado y verificado",
                "Muestras": muestras_interp
            })
        else:
            reporte_final.append({
                "Archivo": nombre_archivo,
                "Estado": "❌ DESCARTADO",
                "Detalle": "; ".join(calidad["motivo_descarte"]),
                "Muestras": 0
            })

    except Exception as e:
        reporte_final.append({
            "Archivo": nombre_archivo,
            "Estado": "❌ EXCEPCIÓN",
            "Detalle": str(e),
            "Muestras": 0
        })

# Convertir a DataFrame y presentar sumario ejecutivo
df_reporte = pd.DataFrame(reporte_final)

print("\n=== RESUMEN DE PROCESAMIENTO DE PIPELINE ===")
if not df_reporte.empty:
    print(df_reporte["Estado"].value_counts())

print(f"\n--- Primeros {procesar_n_muestras} resultados en detalle ---")
display(df_reporte.head(procesar_n_muestras))

Se encontraron 78 archivos JSON en Google Drive.

=== RESUMEN DE PROCESAMIENTO DE PIPELINE ===
Estado
✅ ACEPTADO      72
❌ DESCARTADO     6
Name: count, dtype: int64

--- Primeros 78 resultados en detalle ---


,Archivo,Estado,Detalle,Muestras
0,20260208022404_temp_BV.json,❌ DESCARTADO,Falta el sensor crítico: 'izquierda',0
1,20260303043120_temp_N.json,❌ DESCARTADO,Duración insuficiente en 'izquierda': 4.92s (M...,0
2,20260303043532_temp_BO.json,❌ DESCARTADO,Duración insuficiente en 'izquierda': 3.66s (M...,0
3,20260303044505_temp_BX.json,❌ DESCARTADO,Falta el sensor crítico: 'derecha',0
4,20260303134630_temp_BJ.json,❌ DESCARTADO,Falta el sensor crítico: 'derecha',0
...,...,...,...,...
73,20260820141919_temp_BI.json,✅ ACEPTADO,Formato normalizado y verificado,2723
74,20260820142005_temp_Z.json,✅ ACEPTADO,Formato normalizado y verificado,2860
75,20260820142006_temp_Z.json,✅ ACEPTADO,Formato normalizado y verificado,2860
76,20260820142037_temp_T.json,✅ ACEPTADO,Formato normalizado y verificado,1229


### **Visualizaciones**
El siguiente bloque de código realiza el procesamiento físico completo para una prueba aceptada (con detección automática de fases de levantamiento y sentado basada en la energía de la espina base) y lo compara con un gráfico de anomalías físicas de una prueba descartada.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def detectar_fases_tug(tiempo, senal_binaria, duracion_minima=0.5):
    """Detecta las fases de levantamiento y sentado en la prueba TUG."""
    transiciones = np.diff(senal_binaria.astype(int))
    indices_subida = np.where(transiciones == 1)[0] + 1   # de 0 a 1
    indices_bajada = np.where(transiciones == -1)[0] + 1  # de 1 a 0

    if senal_binaria[0] == 1:
        indices_subida = np.concatenate([[0], indices_subida])
    if senal_binaria[-1] == 1:
        indices_bajada = np.concatenate([indices_bajada, [len(senal_binaria) - 1]])

    fases_movimiento = []
    for i in range(min(len(indices_subida), len(indices_bajada))):
        inicio_mov = indices_subida[i]
        fin_mov = indices_bajada[i]
        if inicio_mov < fin_mov:
            t_inicio = tiempo[inicio_mov]
            t_fin = tiempo[fin_mov]
            duracion = t_fin - t_inicio
            if duracion >= duracion_minima:
                fases_movimiento.append({
                    'inicio_tiempo': t_inicio, 'fin_tiempo': t_fin, 'duracion': duracion,
                    'inicio_indice': inicio_mov, 'fin_indice': fin_mov
                })

    fases = {'levantamiento': None, 'sentado': None, 'todas_las_fases': fases_movimiento}
    if len(fases_movimiento) >= 1:
        fases['levantamiento'] = fases_movimiento[0]
    if len(fases_movimiento) >= 2:
        fases['sentado'] = fases_movimiento[-1]
    return fases


def graficar_fases_tug(tiempo, senal_original, senal_binaria, t_movil, pmovil, fases, titulo="Detección de Fases TUG"):
    plt.figure(figsize=(12, 8))
    plt.plot(tiempo, senal_original, label='Señal Original', alpha=0.7)
    plt.plot(t_movil, pmovil, label='Promedio Móvil', linewidth=2)
    plt.plot(t_movil, senal_binaria, label='Señal Binaria', linewidth=2)

    colores = ['red', 'orange', 'purple', 'brown']
    for i, fase in enumerate(fases['todas_las_fases']):
        plt.axvspan(fase['inicio_tiempo'], fase['fin_tiempo'], alpha=0.3,
                    color=colores[i % len(colores)], label=f'Fase {i+1}: {fase["duracion"]:.2f}s')

    if fases['levantamiento']:
        lev = fases['levantamiento']
        plt.axvspan(lev['inicio_tiempo'], lev['fin_tiempo'], alpha=0.5, color='green',
                    label=f'LEVANTAMIENTO: {lev["duracion"]:.2f}s')
    if fases['sentado']:
        sent = fases['sentado']
        plt.axvspan(sent['inicio_tiempo'], sent['fin_tiempo'], alpha=0.5, color='blue',
                    label=f'SENTADO: {sent["duracion"]:.2f}s')

    plt.xlabel("Tiempo (s)")
    plt.ylabel("Señal")
    plt.title(titulo)
    plt.grid(True)
    plt.legend()
    plt.show()


def imprimir_resumen_fases(fases):
    print("\n" + "=" * 50)
    print("RESUMEN DE FASES DETECTADAS")
    print("=" * 50)
    print(f"Total de fases de movimiento detectadas: {len(fases['todas_las_fases'])}")

    if fases['levantamiento']:
        lev = fases['levantamiento']
        print(f"\n🚶 FASE DE LEVANTAMIENTO:")
        print(f"   Tiempo inicio: {lev['inicio_tiempo']:.2f} s")
        print(f"   Tiempo fin: {lev['fin_tiempo']:.2f} s")
        print(f"   Duración: {lev['duracion']:.2f} s")
    else:
        print("\n❌ No se detectó fase de levantamiento")

    if fases['sentado']:
        sent = fases['sentado']
        print(f"\n🪑 FASE DE SENTADO:")
        print(f"   Tiempo inicio: {sent['inicio_tiempo']:.2f} s")
        print(f"   Tiempo fin: {sent['fin_tiempo']:.2f} s")
        print(f"   Duración: {sent['duracion']:.2f} s")
    else:
        print("\n❌ No se detectó fase de sentado")

    print(f"\n📊 TODAS LAS FASES:")
    for i, fase in enumerate(fases['todas_las_fases']):
        print(f"   Fase {i+1}: {fase['inicio_tiempo']:.2f}s - {fase['fin_tiempo']:.2f}s (Duración: {fase['duracion']:.2f}s)")


# ----------------------- Variables de control -----------------------
path = "20260618075259_temp_BC.json"   # nombre dentro de DRIVE_TUG_PATH o ruta completa
lateral = "espina_base"
signal = "b"
indice_muestra = 0                     # el formato nuevo trae 1 prueba por archivo

try:
    pruebas = cargar_prueba_adaptada(path, fs_destino=50.0)
    clave_muestra, time, asignal = extraer_senal(pruebas, indice_muestra, lateral, signal)

    asignalfiltered = filter_signal(asignal, cutoff_hz=3, fs=50.0, order=20)

    times, energies = energy_spectrum(
        asignalfiltered,
        fs=50.0,
        window_size_samples=50,
        step_size_samples=1,
        apply_window=True
    )

    t_movil, pmovil = promediomovil(energies, ventana=50, fs=50.0)
    binary = binary_threshold(pmovil, threshold=0.2)

    fases_detectadas = detectar_fases_tug(t_movil, binary, duracion_minima=0.5)
    imprimir_resumen_fases(fases_detectadas)
    graficar_fases_tug(time, asignalfiltered, binary, t_movil, pmovil, fases_detectadas,
                       titulo=f"Detección de Fases TUG - {os.path.basename(path)}")

except Exception as e:
    print(f"No se pudo graficar: {e}")

No se pudo graficar: name 'cargar_datos_json_local' is not defined
